# First-level, group-level and ROI analysis of a language-localizer fMRI dataset (Nilearn)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anasshwahen442-code/fmri-first-level-glm/blob/main/fmri_first_level_glm.ipynb)

**What this notebook does**

1. Quality control: framewise displacement per subject (reported, nobody excluded).
2. First-level GLM per subject (SPM HRF, cosine drift, AR(1), 6 motion regressors).
3. Group-level one-sample model with FDR-corrected whole-brain map and a cluster table.
4. ROI analysis with hypotheses written down before the ROI values were extracted:
   t-test, exact sign-flip permutation test, Wilcoxon test, Bonferroni correction, effect sizes with 95% CIs.
5. Robustness check across two smoothing kernels (6 mm and 8 mm).
6. Motion sensitivity check: the primary model is refit with one spike regressor per high-motion volume (FD > 0.5 mm).

**What it is not:** a replication study or a methods contribution. It is a worked example on the 10-subject Nilearn demo dataset.
Preprocessing (realignment, MNI normalisation) was done by the dataset authors, not here.

## 0. Setup

In [ ]:
%pip install -q "nilearn>=0.10.4"

In [ ]:
import itertools, json, platform, sys, warnings, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from scipy import stats

import nilearn
from nilearn import datasets, image, plotting
from nilearn.datasets import fetch_language_localizer_demo_dataset
from nilearn.glm import threshold_stats_img
from nilearn.glm.first_level import FirstLevelModel
from nilearn.glm.second_level import SecondLevelModel
from nilearn.maskers import NiftiMasker
from nilearn.plotting import plot_design_matrix
from nilearn.reporting import get_clusters_table

warnings.filterwarnings("ignore", category=FutureWarning)

# ----------------------------- analysis settings -----------------------------
CONTRAST = "language - string"
SUBJECTS = [f"sub-{i:02d}" for i in range(1, 11)]
MOTION = ["X", "Y", "Z", "RotX", "RotY", "RotZ"]      # motion parameters only
FWHM_MAIN = 8                                           # mm, primary analysis
FWHM_SENS = [6, 8]                                      # mm, robustness check
HIGH_PASS = 1 / 128                                     # Hz
FD_THRESHOLD = 0.5                                      # mm, reported only
ALPHA = 0.05
RUN_WHOLE_BRAIN_PERMUTATION = False                     # slow; optional
SEED = 0
assert FWHM_MAIN in FWHM_SENS

OUT_FIG, OUT_RES = Path("figures"), Path("results")
for p in (OUT_FIG, OUT_RES):
    p.mkdir(exist_ok=True)

print("python", sys.version.split()[0], "| nilearn", nilearn.__version__,
      "| numpy", np.__version__, "| pandas", pd.__version__)

## 1. Data

In [ ]:
fetched = fetch_language_localizer_demo_dataset()
# nilearn >= 0.10.3 returns a Bunch (older/legacy output: a tuple whose first item is the directory)
root = Path(fetched.data_dir if hasattr(fetched, "data_dir") else fetched[0])
print(root)
print(getattr(fetched, "description", "(no dataset description returned by this nilearn version)"))

def load_subject(sub):
    func = root / "derivatives" / sub / "func"
    stem = f"{sub}_task-languagelocalizer"
    bold = func / f"{stem}_desc-preproc_bold.nii.gz"
    with open(func / f"{stem}_desc-preproc_bold.json") as f:
        tr = json.load(f)["RepetitionTime"]
    events = pd.read_csv(root / sub / "func" / f"{stem}_events.tsv", sep="\t")
    conf = pd.read_csv(func / f"{stem}_desc-confounds_regressors.tsv", sep="\t")
    return bold, tr, events, conf

_, tr0, ev0, _ = load_subject(SUBJECTS[0])
print("TR =", tr0, "s")
print(ev0.trial_type.value_counts())

## 2. Quality control: head motion

In [ ]:
def framewise_displacement(conf):
    """Use the confounds file's FD if present, otherwise compute it (rotations in radians -> mm on a 50 mm sphere)."""
    for col in ("framewise_displacement", "FramewiseDisplacement"):
        if col in conf.columns:
            return conf[col].fillna(0).to_numpy()
    mp = conf[MOTION].fillna(0).to_numpy()
    mp = np.column_stack([mp[:, :3], mp[:, 3:] * 50.0])
    return np.r_[0.0, np.abs(np.diff(mp, axis=0)).sum(axis=1)]

qc_rows = []
for sub in SUBJECTS:
    _, tr, events, conf = load_subject(sub)
    fd = framewise_displacement(conf)
    qc_rows.append(dict(subject=sub, TR=tr, n_volumes=len(conf), n_events=len(events),
                        mean_FD_mm=fd.mean(), max_FD_mm=fd.max(),
                        pct_volumes_FD_over_thr=100 * (fd > FD_THRESHOLD).mean()))
qc = pd.DataFrame(qc_rows)
qc.to_csv(OUT_RES / "qc_motion.csv", index=False)
qc.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.2))
ax.bar(qc.subject.str.replace("sub-", ""), qc.mean_FD_mm, color="#4c78a8")
ax.axhline(FD_THRESHOLD, ls="--", c="grey", lw=1)
ax.set_xlabel("subject"); ax.set_ylabel("mean FD (mm)")
ax.set_title("Head motion per subject (no subject excluded)")
fig.tight_layout(); fig.savefig(OUT_FIG / "qc_motion.png", dpi=150); plt.show()

## 3. First-level models

One model per subject and smoothing kernel. Effect-size maps feed the group model and the ROI analysis;
the primary analysis uses the 8 mm kernel, the 6 mm run is a robustness check. Expect several minutes on Colab.

In [ ]:
CACHE = OUT_RES / "effect_maps"          # re-runs reuse these maps instead of refitting 20 models
USE_CACHE = True

def fit_subject(sub, fwhm, extra_confounds=None):
    bold, tr, events, conf = load_subject(sub)
    motion = conf[[c for c in MOTION if c in conf.columns]].fillna(0)
    if extra_confounds is not None:
        motion = pd.concat([motion.reset_index(drop=True), extra_confounds.reset_index(drop=True)], axis=1)
    model = FirstLevelModel(t_r=tr, hrf_model="spm", drift_model="cosine",
                            high_pass=HIGH_PASS, smoothing_fwhm=fwhm, noise_model="ar1")
    return model.fit(str(bold), events=events, confounds=motion)

effects = {fwhm: {} for fwhm in FWHM_SENS}
model_01 = z_01 = None
for fwhm in FWHM_SENS:
    (CACHE / f"fwhm{fwhm}").mkdir(parents=True, exist_ok=True)
    for sub in SUBJECTS:
        f = CACHE / f"fwhm{fwhm}" / f"{sub}_effect.nii.gz"
        need_model_01 = (fwhm == FWHM_MAIN and sub == SUBJECTS[0])
        if USE_CACHE and f.exists() and not need_model_01:
            effects[fwhm][sub] = image.load_img(str(f))
            continue
        m = fit_subject(sub, fwhm)
        effects[fwhm][sub] = m.compute_contrast(CONTRAST, output_type="effect_size")
        effects[fwhm][sub].to_filename(str(f))
        if need_model_01:
            model_01 = m
            z_01 = m.compute_contrast(CONTRAST, output_type="z_score")
    print(f"first-level done: FWHM = {fwhm} mm")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
plot_design_matrix(model_01.design_matrices_[0], ax=ax)
ax.set_title("Design matrix, sub-01")
fig.tight_layout(); fig.savefig(OUT_FIG / "design_matrix_sub-01.png", dpi=150); plt.show()

disp = plotting.plot_glass_brain(z_01, threshold=3.1, colorbar=True,
                                 title="language > string, sub-01 (uncorrected z > 3.1)")
disp.savefig(str(OUT_FIG / "sub-01_glass_brain.png"), dpi=150)
plotting.show()

## 4. Group level (whole brain, exploratory)

In [ ]:
def group_map(effect_dict):
    imgs = [effect_dict[s] for s in SUBJECTS]
    design = pd.DataFrame({"intercept": [1] * len(imgs)})
    slm = SecondLevelModel().fit(imgs, design_matrix=design)
    return slm.compute_contrast("intercept", output_type="z_score")

z_group = group_map(effects[FWHM_MAIN])
thr_img, thr_val = threshold_stats_img(z_group, alpha=ALPHA, height_control="fdr")
print(f"FDR q < {ALPHA} threshold: z > {thr_val:.3f}")

disp = plotting.plot_glass_brain(thr_img, colorbar=True,
                                 title=f"Group (n={len(SUBJECTS)}): {CONTRAST}, FDR q<{ALPHA}")
disp.savefig(str(OUT_FIG / "group_glass_brain_fdr.png"), dpi=150)
plotting.show()

clusters = get_clusters_table(z_group, stat_threshold=thr_val, cluster_threshold=10)
clusters.to_csv(OUT_RES / "group_clusters_fdr.csv", index=False)
clusters.head(15)

In [ ]:
if RUN_WHOLE_BRAIN_PERMUTATION:
    from nilearn.glm.second_level import non_parametric_inference
    design = pd.DataFrame({"intercept": [1] * len(SUBJECTS)})
    out = non_parametric_inference(
        [effects[FWHM_MAIN][s] for s in SUBJECTS], design_matrix=design,
        second_level_contrast="intercept", n_perm=5000, two_sided_test=False,
        random_state=SEED, n_jobs=-1)
    sig = image.threshold_img(out["logp_max_t"], threshold=-np.log10(ALPHA))
    disp = plotting.plot_glass_brain(sig, colorbar=True,
                                     title="Permutation, max-t FWER p<.05 (-log10 p)")
    disp.savefig(str(OUT_FIG / "group_glass_brain_permutation.png"), dpi=150)
    plotting.show()
else:
    print("Whole-brain permutation test skipped (set RUN_WHOLE_BRAIN_PERMUTATION = True to run).")

## 5. Hypotheses for the ROI analysis

ROIs were chosen from the language-localizer literature before any ROI value was extracted.
The whole-brain maps above had already been viewed, and the hypotheses are **not formally pre-registered**
(the git history of this repository is the only time stamp).

* **H1:** the mean effect for `language > string` is positive in the left inferior frontal gyrus (IFG) and in the left superior temporal gyrus (STG).
  Test: one-sample t-test across subjects, Bonferroni-corrected over all 4 ROIs (family = 4).
* **H2 (control):** no reliable effect in the left and right hippocampus.
  A non-significant result is not evidence of absence at n = 10; an equivalence test is added as a post-hoc, exploratory check.

All results are reported below, whether or not they support these hypotheses.
Robustness: exact sign-flip permutation test and Wilcoxon test (no normality assumption), and a second smoothing kernel.

## 6. ROI definitions (Harvard-Oxford atlas)

In [ ]:
cort = datasets.fetch_atlas_harvard_oxford("cort-maxprob-thr25-2mm")
subc = datasets.fetch_atlas_harvard_oxford("sub-maxprob-thr25-2mm")

def labels_containing(atlas, text):
    return [l for l in atlas.labels if text.lower() in l.lower()]

def build_roi(atlas, names, hemisphere=None):
    labels = list(atlas.labels)
    idx = [labels.index(n) for n in names]
    img = image.load_img(atlas.maps)
    arr = img.get_fdata()
    mask = np.isin(arr, idx)
    if hemisphere in ("left", "right"):
        i, j, k = np.indices(arr.shape)
        x = img.affine[0, 0] * i + img.affine[0, 1] * j + img.affine[0, 2] * k + img.affine[0, 3]
        mask &= (x < 0) if hemisphere == "left" else (x > 0)
    return image.new_img_like(img, mask.astype("int8"))

spec = {
    "L IFG":         (cort, labels_containing(cort, "Inferior Frontal Gyrus"), "left"),
    "L STG":         (cort, labels_containing(cort, "Superior Temporal Gyrus"), "left"),
    "L hippocampus": (subc, labels_containing(subc, "Left Hippocampus"), None),
    "R hippocampus": (subc, labels_containing(subc, "Right Hippocampus"), None),
}
for name, (_, labs, hemi) in spec.items():
    print(f"{name:14s} <- {labs} (hemisphere filter: {hemi})")

rois = {name: build_roi(a, labs, hemi) for name, (a, labs, hemi) in spec.items()}
ref = effects[FWHM_MAIN][SUBJECTS[0]]
rois_rs = {n: image.resample_to_img(r, ref, interpolation="nearest") for n, r in rois.items()}
roi_vox = {n: int(image.get_data(r).sum()) for n, r in rois_rs.items()}
assert all(v > 0 for v in roi_vox.values()), roi_vox
print("voxels per ROI on the analysis grid:", roi_vox)

In [ ]:
for name, roi in rois.items():
    d = plotting.plot_roi(roi, title=name, display_mode="ortho")
    d.savefig(str(OUT_FIG / f"roi_{name.replace(' ', '_')}.png"), dpi=150)
plotting.show()

## 7. ROI statistics

In [ ]:
def exact_signflip_p(x):
    """Exact two-sided sign-flip permutation test of mean(x) = 0 (2^n flips, n = 10 -> 1024)."""
    signs = np.array(list(itertools.product([-1, 1], repeat=len(x))))
    null = (signs * x).mean(axis=1)
    return float((np.abs(null) >= abs(x.mean()) - 1e-12).mean())

def roi_stats(vals):
    n, m, sd = len(vals), vals.mean(), vals.std(ddof=1)
    t, p = stats.ttest_1samp(vals, 0)
    lo, hi = stats.t.interval(0.95, n - 1, loc=m, scale=stats.sem(vals))
    try:
        p_w = stats.wilcoxon(vals).pvalue
    except ValueError:
        p_w = np.nan
    return dict(mean=m, ci_low=lo, ci_high=hi, t=t, df=n - 1, p=p, d_z=m / sd,
                p_signflip=exact_signflip_p(vals), p_wilcoxon=p_w, n_positive=int((vals > 0).sum()))

def roi_values(effect_dict, roi_img):
    masker = NiftiMasker(mask_img=roi_img, standardize=False).fit()
    return np.array([masker.transform(effect_dict[s]).mean() for s in SUBJECTS])

rows, per_subject = [], []
for fwhm in FWHM_SENS:
    for name, roi in rois_rs.items():
        vals = roi_values(effects[fwhm], roi)
        r = roi_stats(vals); r.update(ROI=name, FWHM_mm=fwhm); rows.append(r)
        per_subject += [dict(ROI=name, FWHM_mm=fwhm, subject=s, effect=v) for s, v in zip(SUBJECTS, vals)]

res = pd.DataFrame(rows)
k = len(rois_rs)
for col in ("p", "p_signflip", "p_wilcoxon"):
    res[col + "_bonf"] = (res[col] * k).clip(upper=1)
per_subject = pd.DataFrame(per_subject)

cols = ["ROI", "FWHM_mm", "mean", "ci_low", "ci_high", "t", "df", "p", "p_bonf",
        "d_z", "p_signflip_bonf", "p_wilcoxon_bonf", "n_positive"]
res = res[cols]
res.to_csv(OUT_RES / "roi_results.csv", index=False)
per_subject.to_csv(OUT_RES / "roi_per_subject.csv", index=False)
res.round(4)

In [ ]:
main = res[res.FWHM_mm == FWHM_MAIN].set_index("ROI")
h1 = all((main.loc[r, "p_bonf"] < ALPHA) and (main.loc[r, "mean"] > 0) for r in ("L IFG", "L STG"))
print(f"H1 (L IFG and L STG positive, Bonferroni p < {ALPHA}, t-test, {FWHM_MAIN} mm):", "supported" if h1 else "NOT supported")
for r in ("L IFG", "L STG"):
    ok = {c: bool(main.loc[r, c] < ALPHA) for c in ("p_bonf", "p_signflip_bonf", "p_wilcoxon_bonf")}
    print(f"  {r}: significant after Bonferroni ->", ok)
robust = res.pivot(index="ROI", columns="FWHM_mm", values="p_bonf")
print("\nBonferroni p (t-test) by smoothing kernel:\n", robust.round(4))

In [ ]:
# Exploratory equivalence test (TOST) for the hippocampal controls.
# Post hoc choice: equivalence bound = +/- one third of the mean left-STG effect (arbitrary units).
bound = main.loc["L STG", "mean"] / 3

def tost_p(vals, bound):
    n, m, se = len(vals), vals.mean(), stats.sem(vals)
    p_lower = 1 - stats.t.cdf((m + bound) / se, n - 1)   # H0: mean <= -bound
    p_upper = stats.t.cdf((m - bound) / se, n - 1)       # H0: mean >= +bound
    return max(p_lower, p_upper)

for r in ("L hippocampus", "R hippocampus"):
    vals = roi_values(effects[FWHM_MAIN], rois_rs[r])
    print(f"{r}: mean = {vals.mean():.4f}, TOST p = {tost_p(vals, bound):.3f} (bound = +/-{bound:.4f})")
print("TOST p < .05 would indicate the effect lies inside the bound; otherwise equivalence is not shown at this n.")

In [ ]:
sub = per_subject[per_subject.FWHM_mm == FWHM_MAIN]
names = list(rois_rs)
fig, ax = plt.subplots(figsize=(6.5, 4))
jit = np.random.default_rng(SEED).uniform(-0.12, 0.12, size=len(SUBJECTS))
for i, n in enumerate(names):
    v = sub[sub.ROI == n].effect.to_numpy()
    ax.scatter(np.full(len(v), i) + jit, v, s=22, color="#4c78a8", alpha=0.7, zorder=3)
    ax.errorbar(i, main.loc[n, "mean"],
                yerr=[[main.loc[n, "mean"] - main.loc[n, "ci_low"]], [main.loc[n, "ci_high"] - main.loc[n, "mean"]]],
                fmt="o", color="black", capsize=4, zorder=4)
ax.axhline(0, color="grey", lw=1)
ax.set_xticks(range(len(names))); ax.set_xticklabels(names, rotation=15)
ax.set_ylabel("mean contrast estimate (a.u.)")
ax.set_title(f"{CONTRAST}: ROI means, 95% CI, individual subjects ({FWHM_MAIN} mm)")
fig.tight_layout(); fig.savefig(OUT_FIG / "roi_effects.png", dpi=150); plt.show()

## 7b. Motion sensitivity check (spike regressors)

The primary models only contain six motion parameters. Here the main-kernel model is refit with one extra
regressor per volume whose framewise displacement exceeds the threshold (volume censoring in the GLM),
and the ROI table is recomputed. Agreement with the primary result is evidence that head motion is not driving it.

In [ ]:
RUN_SCRUB_SENS = True

def spike_regressors(sub):
    _, _, _, conf = load_subject(sub)
    fd = framewise_displacement(conf)
    idx = np.flatnonzero(fd > FD_THRESHOLD)
    return pd.DataFrame(np.eye(len(fd))[:, idx], columns=[f"spike_{i}" for i in idx])

if RUN_SCRUB_SENS:
    scrub_effects = {}
    for sub in SUBJECTS:
        sp = spike_regressors(sub)
        m = fit_subject(sub, FWHM_MAIN, extra_confounds=sp if sp.shape[1] else None)
        scrub_effects[sub] = m.compute_contrast(CONTRAST, output_type="effect_size")
        print(f"{sub}: {sp.shape[1]} spike regressors")

    rows = []
    for name, roi in rois_rs.items():
        vals = roi_values(scrub_effects, roi)
        r = roi_stats(vals); r.update(ROI=name); rows.append(r)
    scrub = pd.DataFrame(rows)
    scrub["p_bonf"] = (scrub["p"] * len(rois_rs)).clip(upper=1)
    scrub = scrub[["ROI", "mean", "ci_low", "ci_high", "t", "p", "p_bonf", "p_signflip", "p_wilcoxon", "n_positive"]]
    scrub.to_csv(OUT_RES / "roi_results_spike_regressors.csv", index=False)
    cmp = pd.DataFrame({"mean_primary": main["mean"], "mean_spikes": scrub.set_index("ROI")["mean"],
                        "p_bonf_primary": main["p_bonf"], "p_bonf_spikes": scrub.set_index("ROI")["p_bonf"]})
    print(cmp.round(4))
else:
    print("Spike-regressor sensitivity analysis skipped.")

## 8. Interpretation (write this in your own words after running)

Suggested checklist: which ROIs passed Bonferroni correction under all three tests; whether the 6 mm and 8 mm kernels agree;
whether the spike-regressor check (section 7b) agrees; how wide the confidence intervals are; and that the hippocampal result
is "no reliable effect detected", not "no effect".

The next cell only prints the numbers; the interpretation itself is yours to write.

In [ ]:
summary = main[["mean", "ci_low", "ci_high", "t", "p_bonf", "p_signflip_bonf", "p_wilcoxon_bonf", "d_z", "n_positive"]]
print(f"Primary analysis ({FWHM_MAIN} mm, n = {len(SUBJECTS)}), mean contrast estimate in arbitrary units:\n")
print(summary.round(3).to_string())

## 9. Limitations

* Ten subjects of a demo dataset; effect sizes (d_z) at this n are imprecise and likely inflated.
* Preprocessing was done by the dataset authors; the primary model regresses out only six motion parameters (no volume scrubbing); section 7b adds a spike-regressor sensitivity check.
* "Contrast estimate" values are in arbitrary units (not percent signal change).
* ROIs come from a max-probability atlas thresholded at 25%; the left-hemisphere split uses the sign of the MNI x-coordinate.
* Hypotheses were written after viewing whole-brain maps and are not formally pre-registered; the equivalence bound is a post hoc choice.
* One task contrast, one dataset, no replication.

## 10. Reproducibility and export

In [ ]:
env = dict(python=sys.version, platform=platform.platform(), nilearn=nilearn.__version__,
           numpy=np.__version__, pandas=pd.__version__, scipy=__import__("scipy").__version__,
           matplotlib=matplotlib.__version__,
           settings=dict(contrast=CONTRAST, subjects=SUBJECTS, fwhm_main=FWHM_MAIN, fwhm_sens=FWHM_SENS,
                         high_pass_hz=HIGH_PASS, hrf="spm", drift="cosine", noise="ar1",
                         motion_regressors=MOTION, alpha=ALPHA, seed=SEED, dataset_dir=str(root)))
(OUT_RES / "environment.json").write_text(json.dumps(env, indent=2))

# exact versions used for this run -> copy into requirements.txt to pin them
from importlib.metadata import version
pins = [f"{p}=={version(p)}" for p in ("nilearn", "numpy", "pandas", "scipy", "matplotlib")]
(OUT_RES / "requirements_frozen.txt").write_text("\n".join(pins) + "\n")
print("\n".join(pins))

with zipfile.ZipFile("fmri_glm_outputs.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for folder in (OUT_FIG, OUT_RES):
        for f in sorted(folder.rglob("*")):
            if f.is_file() and "effect_maps" not in f.parts:
                z.write(f)
print("saved:", *sorted(str(p) for p in list(OUT_FIG.glob("*")) + list(OUT_RES.glob("*"))), sep="\n  ")

try:
    from google.colab import files
    files.download("fmri_glm_outputs.zip")
except ImportError:
    print("Not running in Colab: files are in ./figures and ./results")